# 📈 Life Expectancy Regression Project

## Overview
This project predicts continuous **Life Expectancy** across countries using the World Health Organization (WHO) dataset.

### Workflow:
1. **Data Ingestion**: Load the raw WHO Life Expectancy dataset.
2. **Exploratory Data Analysis (EDA)**: Inspect column headers, preview sample data, review data types, summary statistics, and missing values.
3. **Data Cleaning & Standardization**: Clean column headers into lowercase `snake_case`, check for duplicates, calculate missing value percentages, and drop redundant columns (`schooling`).
4. **Correlation Analysis**: Measure linear correlations between predictor features and continuous `life_expectancy`.
5. **Handling Target Missing Values & Train-Test Split**: Handle missing target entries with median imputation and partition into training and testing sets (80/20 split).
6. **Feature Segregation**: Distinguish categorical and numerical feature sets.
7. **Feature Selection (RFE)**: Apply Recursive Feature Elimination with Ridge estimator to identify the top 7 features.
8. **Preprocessing Pipelines & Model Assembly**: Build numerical (`KNNImputer` + `StandardScaler`) and categorical (`SimpleImputer` + `OneHotEncoder`) pipelines joined with `Ridge(alpha=0.01)` in a unified pipeline.
9. **Model Training & Inference**: Fit on training set and generate predictions on holdout test set.
10. **Evaluation Metrics**: Assess accuracy via MAE, MSE, RMSE, and R² Score.
11. **Residual Diagnostics**: Analyze residual plot to verify homoscedasticity and zero-mean error distribution.
12. **Hyperparameter Tuning (`GridSearchCV`)**: Optimize Ridge regularisation strength `alpha` with 5-fold cross-validation.

## 1. Import Required Libraries
Import libraries for data handling, exploratory analysis, visualization, preprocessing, and regression modeling.

In [ ]:
# ==============================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ==============================================================================

# Core libraries for numerical computations and data manipulation
import numpy as np
import pandas as pd

# Data visualization libraries
import seaborn as sns
import matplotlib.pyplot as plt

# Scikit-Learn: Imputation strategies
from sklearn.impute import SimpleImputer, KNNImputer

# Scikit-Learn: Feature preprocessing, scaling, and encoding
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Scikit-Learn: Feature selection
from sklearn.feature_selection import RFE

# Scikit-Learn: Model selection, cross-validation, and regression models
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, LinearRegression

# Scikit-Learn: Regression evaluation metrics
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## 2. Load Dataset
Read the WHO Life Expectancy CSV data into a pandas DataFrame.

In [ ]:
# ==============================================================================
# STEP 2: LOAD DATASET
# ==============================================================================
# Read the WHO Life Expectancy CSV data into a pandas DataFrame
df = pd.read_csv("Life Expectancy Data.csv")

## 3. Exploratory Data Analysis (EDA)
Inspect column headers, preview records, examine schema and data types, and check summary statistics and missing values.

In [ ]:
# Inspect initial column headers
df.columns

In [ ]:
# Preview the first 5 records of the dataset
df.head()

In [ ]:
# Preview the last 5 records of the dataset
df.tail()

In [ ]:
# Display concise summary: column data types, non-null counts, and memory usage
df.info()

In [ ]:
# Statistical summary for categorical variables ('Country' and 'Status')
df.describe(include='object')

In [ ]:
# Statistical summary for numerical features (mean, std, min, quartiles, max)
df.describe()

In [ ]:
# Compute and sort count of missing (null) values per column
df.isnull().sum().sort_values(ascending=False)

## 4. Data Cleaning & Standardization
Standardize column names to lower `snake_case`, check for duplicates, profile missing percentages, and remove redundant features (`schooling`).

In [ ]:
# ==============================================================================
# STEP 4: DATA CLEANING & COLUMN STANDARDIZATION
# ==============================================================================
# Standardize column names:
# 1. Strip leading and trailing whitespace
# 2. Convert all characters to lowercase
# 3. Replace spaces with underscores
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)

In [ ]:
# Verify duplicate rows across the entire dataset (0 duplicates found)
df.duplicated().sum()

In [ ]:
# Inspect cardinality (count of unique distinct values) per column
df.nunique().sort_values()

In [ ]:
# Build a summary DataFrame showing raw count and percentage of missing values
missing = pd.DataFrame({
    'Missing': df.isnull().sum(),
    'Percentage': df.isnull().mean() * 100
})

# Display missing data sorted by percentage descending
missing.sort_values(by='Percentage', ascending=False)

In [ ]:
# Drop 'schooling' feature due to high multicollinearity / redundancy
df = df.drop(columns='schooling')

In [ ]:
# Confirm duplicate count remains 0 after dropping 'schooling'
df.duplicated().sum()

In [ ]:
# Re-evaluate missing value counts across remaining columns
df.isnull().sum()

## 5. Correlation Analysis
Compute Pearson correlation between numerical features and the continuous target `life_expectancy`.

In [ ]:
# ==============================================================================
# STEP 5: CORRELATION ANALYSIS (CONTINUOUS TARGET)
# ==============================================================================
# Calculate Pearson correlation of numerical features against 'life_expectancy'
corr = df.corr(numeric_only=True)['life_expectancy']

print(corr.sort_values(ascending=False))

## 6. Target Imputation & Train-Test Split
Impute missing values in `life_expectancy` using the median, separate feature matrix $X$ and target vector $y$, and partition into 80% train and 20% test sets.

In [ ]:
# ==============================================================================
# STEP 6: HANDLING TARGET MISSING VALUES & TRAIN-TEST SPLIT
# ==============================================================================
# Impute missing values in continuous target 'life_expectancy' with the median
df['life_expectancy'] = df['life_expectancy'].fillna(df['life_expectancy'].median())

# Separate features (X) and target (y)
X = df.drop(columns=['life_expectancy'])
y = df['life_expectancy']

# Perform an 80% train / 20% test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

## 7. Feature Segregation & Preprocessing Pipelines
Define categorical and numerical column subsets, and build pipelines with `KNNImputer`, `StandardScaler`, and `OneHotEncoder`.

In [ ]:
# Categorical columns requiring imputation and one-hot encoding
cat_cols = ['status', 'country']

# Numerical columns requiring KNN-based imputation and standard scaling
num_cols = [
    'year', 'adult_mortality', 'infant_deaths', 'alcohol',
    'percentage_expenditure', 'hepatitis_b', 'measles', 'bmi',
    'under-five_deaths', 'polio', 'total_expenditure', 'diphtheria',
    'hiv/aids', 'gdp', 'population', 'thinness__1-19_years',
    'thinness_5-9_years', 'income_composition_of_resources'
]

In [ ]:
# Key Correlations with Continuous Life Expectancy:
# - Positive associations:
#   income_composition_of_resources : +0.7248
#   bmi                             : +0.5677
#   diphtheria                      : +0.4795
#   polio                           : +0.4656
#   gdp                             : +0.4615
#   alcohol                         : +0.4049
# - Negative associations:
#   thinness_5-9_years              : -0.4716
#   thinness__1-19_years            : -0.4772
#   hiv/aids                        : -0.5566
#   adult_mortality                 : -0.6964

## 8. Feature Selection: Recursive Feature Elimination (RFE)
Configure RFE using a Ridge regression estimator (`alpha=0.01`) to iteratively select the top 7 most informative features.

In [ ]:
# ==============================================================================
# STEP 8: FEATURE SELECTION VIA RECURSIVE FEATURE ELIMINATION (RFE)
# ==============================================================================
# Initialize RFE using Ridge(alpha=0.01) estimator to select the top 7 features
rfe = RFE(
    estimator=Ridge(0.01),
    n_features_to_select=7,
    step=1
)

## 9. Preprocessing Pipelines & Full Model Assembly
Construct sub-pipelines for numerical features (`KNNImputer` + `StandardScaler`) and categorical features (`SimpleImputer` + `OneHotEncoder`), combined via `ColumnTransformer` and assembled into an end-to-end `Pipeline` with `Ridge(alpha=0.01)`.

In [ ]:
# ==============================================================================
# STEP 9: PREPROCESSING PIPELINES & COMPLETE MODEL ASSEMBLY
# ==============================================================================

# Numerical Pipeline: Impute missing values with KNN (k=5) and standardize scales
num_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Impute missing entries with most frequent value and one-hot encode
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers using ColumnTransformer
preprocess = ColumnTransformer([
    ('cat', cat_pipeline, cat_cols),
    ('num', num_pipeline, num_cols)
])

# Assemble complete end-to-end Pipeline: Preprocessing -> Ridge Regressor (alpha=0.01)
model = Pipeline([
    ('preprocess', preprocess),
    ('model', Ridge(0.01))
])

## 10. Model Training
Fit the complete pipeline on the training dataset (`X_train`, `y_train`).

In [ ]:
# ==============================================================================
# STEP 10: TRAIN PIPELINE ON TRAINING DATA
# ==============================================================================
# Fit the full pipeline (imputation, scaling, one-hot encoding, and Ridge regression)
model.fit(X_train, y_train)

## 11. Model Inference
Generate life expectancy predictions on the unseen holdout test set (`X_test`).

In [ ]:
# ==============================================================================
# STEP 11: GENERATE PREDICTIONS ON HOLD-OUT TEST SET
# ==============================================================================
# Predict continuous life expectancy values for unseen test instances
y_pred = model.predict(X_test)

## 12. Model Evaluation (Regression Metrics)
Evaluate regression model performance using **MAE** (Mean Absolute Error), **MSE** (Mean Squared Error), **RMSE** (Root Mean Squared Error), and the **R² Score** (Coefficient of Determination).

In [ ]:
# ==============================================================================
# STEP 12: MODEL EVALUATION (REGRESSION PERFORMANCE METRICS)
# ==============================================================================
# Compute Mean Absolute Error (MAE): Average magnitude of prediction errors
mae = mean_absolute_error(y_test, y_pred)

# Compute Mean Squared Error (MSE): Penalizes larger errors quadratically
mse = mean_squared_error(y_test, y_pred)

# Compute Root Mean Squared Error (RMSE): Standard deviation of prediction residuals (years)
rmse = np.sqrt(mse)

# Compute Coefficient of Determination (R²): Proportion of variance explained by model
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

## 13. Residual Diagnostics
Examine the residual plot ($	ext{Residuals} = y_{	ext{test}} - hat{y}$) to check error symmetry, homoscedasticity, and ensure no non-linear patterns remain uncaptured.

In [ ]:
# ==============================================================================
# STEP 13: RESIDUAL ANALYSIS (DIAGNOSTIC PLOT)
# ==============================================================================
# Calculate prediction residuals: actual - predicted
residuals = y_test - y_pred

# Plot Residuals vs. Predicted Values to evaluate homoscedasticity and error distribution
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.5, color='royalblue', edgecolor='k', s=40)
plt.axhline(y=0, linestyle='--', color='red', linewidth=1.2)
plt.xlabel('Predicted Life Expectancy (Years)')
plt.ylabel('Residuals (Actual - Predicted)')
plt.title('Residual Plot (Ridge Regression)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 14. Hyperparameter Optimization (`GridSearchCV`)
Perform 5-fold cross-validation to search for the optimal L2 regularisation parameter (`model__alpha`) minimizing Mean Squared Error.

In [ ]:
# ==============================================================================
# STEP 14: HYPERPARAMETER TUNING VIA GRIDSEARCHCV
# ==============================================================================
# Define hyperparameter grid for Ridge regularization penalty (alpha)
param = {
    'model__alpha': [0.001, 0.01, 0.1, 1.0, 0.5, 0.7, 0.08, 0.9],
}

# 5-fold cross-validation optimizing Negative Mean Squared Error across all CPU cores
grid = GridSearchCV(
    estimator=model,
    param_grid=param,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# Run hyperparameter search on training data
grid.fit(X_train, y_train)

# Output best alpha parameter and optimal cross-validation score
print("Best parameter: ", grid.best_params_)
print("Best Score: ", grid.best_score_)